# Financial Health Dashboard: Tesla vs BYD vs Ford
Analyzing EV transition impact across three automotive archetypes
- Data Source: Yahoo Finance via yfinance
- Period: 2019-2024

In [1]:
# Install if needed
# !pip install yfinance pandas

import yfinance as yf
import pandas as pd

# Define companies
tickers = {
    "Tesla": "TSLA",
    "BYD": "BYDDY",  # BYD's US-listed ADR
    "Ford": "F"
}

In [2]:
# Pull financial statements for all three companies
data = {}

for name, ticker in tickers.items():
    company = yf.Ticker(ticker)
    data[name] = {
        "income": company.financials,
        "balance_sheet": company.balance_sheet,
        "cashflow": company.cashflow
    }
    print(f"{name} data pulled successfully")

Tesla data pulled successfully
BYD data pulled successfully
Ford data pulled successfully


In [3]:
# Inspect Tesla's income statement first
print("=== TESLA INCOME STATEMENT ===")
print(data["Tesla"]["income"])
print("\n=== COLUMNS ===")
print(data["Tesla"]["income"].index.tolist())

=== TESLA INCOME STATEMENT ===
                                                      2025-12-31  \
Tax Effect Of Unusual Items                        -1.333800e+08   
Tax Rate For Calcs                                  2.700000e-01   
Normalized EBITDA                                   1.225800e+10   
Total Unusual Items                                -4.940000e+08   
Total Unusual Items Excluding Goodwill             -4.940000e+08   
Net Income From Continuing Operation Net Minori...  3.794000e+09   
Reconciled Depreciation                             6.148000e+09   
Reconciled Cost Of Revenue                          7.773300e+10   
EBITDA                                              1.176400e+10   
EBIT                                                5.616000e+09   
Net Interest Income                                 1.342000e+09   
Interest Expense                                    3.380000e+08   
Interest Income                                     1.680000e+09   
Normalized Income

In [ ]:
# Inspect Tesla's cash flow statement and balance sheet columns
print("=== TESLA CASHFLOW COLUMNS ===")
print(data["Tesla"]["cashflow"].index.tolist())

print("\n=== TESLA BALANCE SHEET COLUMNS ===")
print(data["Tesla"]["balance_sheet"].index.tolist())

=== TESLA CASHFLOW COLUMNS ===
['Free Cash Flow', 'Repayment Of Debt', 'Issuance Of Debt', 'Issuance Of Capital Stock', 'Capital Expenditure', 'Interest Paid Supplemental Data', 'Income Tax Paid Supplemental Data', 'End Cash Position', 'Beginning Cash Position', 'Effect Of Exchange Rate Changes', 'Changes In Cash', 'Financing Cash Flow', 'Cash Flow From Continuing Financing Activities', 'Net Other Financing Charges', 'Proceeds From Stock Option Exercised', 'Net Common Stock Issuance', 'Common Stock Issuance', 'Net Issuance Payments Of Debt', 'Net Long Term Debt Issuance', 'Long Term Debt Payments', 'Long Term Debt Issuance', 'Investing Cash Flow', 'Cash Flow From Continuing Investing Activities', 'Net Other Investing Changes', 'Net Investment Purchase And Sale', 'Sale Of Investment', 'Purchase Of Investment', 'Net Business Purchase And Sale', 'Sale Of Business', 'Purchase Of Business', 'Net Intangibles Purchase And Sale', 'Sale Of Intangibles', 'Purchase Of Intangibles', 'Net PPE Purch

In [5]:
# Calculate the three core metrics for all companies
metrics = {}

for name in tickers.keys():
    income = data[name]["income"]
    cashflow = data[name]["cashflow"]
    balance = data[name]["balance_sheet"]
    
    # Net Profit Margin = Net Income / Total Revenue
    npm = (income.loc["Net Income"] / income.loc["Total Revenue"]) * 100
    
    # Free Cash Flow (already calculated by yfinance)
    fcf = cashflow.loc["Free Cash Flow"]
    
    # Return on Assets = Net Income / Total Assets
    roa = (income.loc["Net Income"] / balance.loc["Total Assets"]) * 100
    
    # Combine into a dataframe
    metrics[name] = pd.DataFrame({
        "Net Profit Margin (%)": npm,
        "Free Cash Flow": fcf,
        "Return on Assets (%)": roa
    })
    
    # Sort by date ascending
    metrics[name] = metrics[name].sort_index()
    
    print(f"\n=== {name} ===")
    print(metrics[name])


=== Tesla ===
            Net Profit Margin (%)  Free Cash Flow  Return on Assets (%)
2021-12-31                    NaN             NaN                   NaN
2022-12-31              15.446466    7.552000e+09             15.282130
2023-12-31              15.499158    4.357000e+09             14.067981
2024-12-31               7.298598    3.581000e+09              5.840911
2025-12-31               4.000970    6.220000e+09              2.753146

=== BYD ===
            Net Profit Margin (%)  Free Cash Flow  Return on Assets (%)
2021-12-31                    NaN             NaN                   NaN
2022-12-31               3.919828    4.338080e+10              3.365817
2023-12-31               4.987555    4.763152e+10              4.420707
2024-12-31               5.180056    3.609410e+10              5.138705
2025-12-31               4.057269   -9.767231e+10              3.691062

=== Ford ===
            Net Profit Margin (%)  Free Cash Flow  Return on Assets (%)
2021-12-31            

In [ ]:
# Drop NaN rows
for name in metrics:
    metrics[name] = metrics[name].dropna()

# Drop rows from partial years
for name in metrics:
    metrics[name] = metrics[name][metrics[name].index.year < 2025]

print("Cleaned date ranges:")
for name in metrics:
    print(f"{name}: {metrics[name].index.tolist()}")